# Experiment 2 — XGBoost Class Weights (scale_pos_weight)

Tells XGBoost that errors on minority class cost more.
No data modification. Fastest technique.
scale_pos_weight = count(background) / count(signal) per version.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

BASE_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.1,
                   random_state=42, eval_metric='auc', verbosity=0,
                   use_label_encoder=False)
print("Experiment 2 — XGBoost Class Weights")

Experiment 2 — XGBoost Class Weights


In [2]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp2-XGB] Training Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    n_bg  = (y_train==0).sum(); n_sig = (y_train==1).sum()
    spw   = n_bg / n_sig
    print(f"[Exp2-XGB] scale_pos_weight={spw:.2f} (bg={n_bg:,}, sig={n_sig:,})")

    model = XGBClassifier(**{**BASE_PARAMS, 'scale_pos_weight': spw})
    t0    = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    probs = model.predict_proba(X_test)[:,1]
    preds = model.predict(X_test)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version'] = v
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp2_xgb_version_{v}.npy'), probs)
    print_metrics(metrics, label=f"Exp2-XGB Version {v}")

print("\n[Exp2-XGB] Complete.")


[Exp2-XGB] Training Version A...
[Exp2-XGB] scale_pos_weight=0.89 (bg=188,238, sig=211,762)
[Exp2-XGB Version A] AUC=0.8212 | F1=0.7489 | Signal_Sig=178.6446 | Train_Time=242.52s

[Exp2-XGB] Training Version B...
[Exp2-XGB] scale_pos_weight=10.00 (bg=188,237, sig=18,823)
[Exp2-XGB Version B] AUC=0.8066 | F1=0.3739 | Signal_Sig=26.0540 | Train_Time=608.96s

[Exp2-XGB] Training Version C...
[Exp2-XGB] scale_pos_weight=50.01 (bg=188,237, sig=3,764)
[Exp2-XGB Version C] AUC=0.7496 | F1=0.1508 | Signal_Sig=5.3881 | Train_Time=569.89s

[Exp2-XGB] Complete.


In [3]:
results_df = pd.DataFrame(all_results)[['Version','AUC','F1','Signal_Significance','Train_Time_sec']]
save_path  = os.path.join(RESULTS_DIR, 'experiment2_class_weights_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      AUC       F1  Signal_Significance  Train_Time_sec
      A 0.821212 0.748854           178.644641          242.52
      B 0.806582 0.373875            26.053977          608.96
      C 0.749604 0.150767             5.388093          569.89

✓ Saved → ..\results\experiment2_class_weights_xgb.csv
